# Tabula Rasa — Socratic vs Standard Benchmark (GPU)

Compares standard training vs Socratic self-improvement on a GPU.
On a T4 GPU: ~2.5 min per seed instead of ~9 hours on CPU.

## Setup
Upload `deploy.zip` to Colab first, then run the cells below.

In [ ]:
# 1. Extract and install
!unzip -q deploy.zip -d /content/tabula-rasa
%cd /content/tabula-rasa
!pip install -q torch numpy tqdm
import sys; sys.path.insert(0, 'src')
import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# 2. Quick smoke test (verify GPU works)
!python3 train_specialist.py add --quick
print('GPU training works!')

In [ ]:
# 3. Run the Socratic benchmark
# This runs 2 seeds of 5000 steps each (standard + Socratic per seed)
# On T4 GPU: ~25s per run, ~4 min total
!python3 scripts/socratic_benchmark.py

In [ ]:
# 4. View results
import json
try:
    with open('results/socratic_comparison.json') as f:
        data = json.load(f)
    print(f"Config: {data['config']}")
    print(f"Summary: {data['summary']}")
    print()
    print(f"{'Seed':<8} {'Variant':<12} {'Op':<6} {'Accuracy':<12} {'Time':<12}")
    for r in data['results']:
        print(f"{r['seed']:<8} {r['variant']:<12} {r['op']:<6} {r['accuracy']:<10.1f}% {r['time_seconds']/60:<10.0f}m")
except FileNotFoundError:
    print("No results yet. Run the benchmark first.")

In [ ]:
# 5. Full 5-seed benchmark (recommended for paper-quality results)
# On T4 GPU: ~25s per run x 10 runs = ~4 minutes
#!python3 scripts/socratic_benchmark.py --seeds 5 --steps 30000
# Uncomment the line above to run the full comparison

## Results Interpretation

The benchmark compares:
- **Standard**: `train_specialist.py add --steps N`
- **Socratic**: `train_specialist.py add --steps N --socratic`

The Socratic refinement runs **after** standard training completes,
so both runs have identical training budgets. The difference is
the post-training self-improvement phase.

In [ ]:
# 6. Save results to Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')
!cp results/socratic_comparison.json /content/drive/MyDrive/
!cp specialists/math/add/best.pt /content/drive/MyDrive/socratic_best.pt 2>/dev/null || true
print('Results saved to Google Drive')